Idea is to build an automated analyser that, given a CSV containing a single control loop’s time series (PV, SP, OP), will:

detect and classify loop behaviours (oscillation, noise, disturbances, variance) and estimate probability of stiction,

generate at least two automatic textual inferences about the loop, and

output a structured summary for each loop (table / JSON).

# Oscillation Model (3-class)
Key features:

peak_ratio

f_peak

autocorr_periodicity_score

# Noise Model (3-class: No / Could be / Yes)
Feature:

hf_ratio (high-frequency power fraction)

# Disturbance Model (3-class)
Feature:

spike_count

avg_spike_mag

#Stiction is probability 





Oscillation      → 3-class (No / Maybe / Yes)
Noise            → 3-class (No / Could be / Yes)
Disturbances     → 3-class (No / Maybe / Yes)
Stiction         → Probability + (derived 3-class if needed)


In [2]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 10.7 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 9.8 MB/s  0:00:30m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]


In [2]:
! pip install numpy pandas scipy scikit-learn

  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 13.6 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.2-py3-none-any.whl (308 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [12]:
pip install openpyxl matplotlib seaborn

  Using cached contourpy-1.3.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.9-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.3 kB)
  Using cached pillow-12.0.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.2.5-py3-none-any.whl.metadata (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 30.1 MB/s  0:00:00 eta 0:00:01
Using cached contourpy-1.3.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (363 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 25.2 MB/s  0:00:00
Using cached kiwisolver-1.4.9-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (1.5 MB)
Using cached pillow-12.0.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.0 MB)
Using cached pyparsing-3.2.5-py3-none-an

In [1]:
import numpy as np
import pandas as pd
import glob
import os
from pathlib import Path
from scipy.signal import welch
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import seaborn as sns
import matplotlib.pyplot as plt

LOOP_DIR = "/Training_Loops"
LABEL_FILE = "Loop_description.xlsx"


In [ ]:
# --- Configurations
LOOPS_DIR = "/Training_Loops"
OUT_JSON = "/data/loops_summary.json"

# list of possible names 
PV_NAMES = ["PV", "pv", "pv_1"]
SP_NAMES = ["SP", "sp", "sp_1"]
OP_NAMES = ["OP", "op", "op_1"]



In [3]:
loop_desc = pd.read_excel("Loop description.xlsx")

loop_desc.head(4)

,Loop No,Oscillatory?,Noisy,Disturbance,Comments,Sticky Valve probability
0,1,No,Yes,No,Only PV,1.00
1,2,Yes,No,No,Looks intermittenly oscillatory,38.04
2,3,Looks oscillatory,No,Yes at PV,Seems to be like stiction,5.00
3,4,Highly Oscillatory,No,No,looks like stiction,50.19


In [6]:
print(f'Oscillatory ones are : {loop_desc['Oscillatory?'].unique().tolist()} ')

print("------------------------")

print(f'NOisy ones are : {loop_desc['Noisy'].unique().tolist()} ')


print("------------------------")

print(f'Disturbance ones are : {loop_desc['Disturbance'].unique().tolist()} ')

Oscillatory ones are : ['No', 'Yes', 'Looks oscillatory', 'Highly Oscillatory', 'Oscillatory', 'Yes. Oscillatory', 'Yes , Long oscillation', 'Yes ', 'Yes, small oscillation', 'Yes, oscillatory', 'yes', 'variance high', 'Yes,', 'Oscillating SP', 'Yes, long oscillation', 'small oscillation', 'Yes, highly oscillatory', 'Noisy', 'Yes (Intermittent)', 'Yes + high variance', 'high variance', 'Some ripples around SP', 'variance', 'intermittently oscillatory', 'Highly intermittenly oscillatory'] 
------------------------
NOisy ones are : ['Yes', 'No', 'Could be', 'No ', 'may be', 'yes', 'no', nan] 
------------------------
Disturbance ones are : ['No', 'Yes at PV', 'no', 'Yes ', 'may be', nan, 'NO', 'Controller tuning / Disturbance', 'Yes', 'No '] 


In [ ]:
# cleaning

Detect oscillations (No / Maybe / Yes)

Detect noise (No / Maybe / Yes)

Detect disturbances (No / Maybe / Yes)

Evaluate variance (Low / Medium / High)

Estimate sticky valve probability (0–1)

Generate final inference text

Work on hidden future test data with zero manual intervention

In [ ]:
import re
import pandas as pd
import numpy as np

def clean_text(x):
    if isinstance(x, str):
        return x.strip().lower()
    return ""

def map_oscillation(text):
    t = clean_text(text)
    if t == "":
        return 0
    
    # Strong oscillation indicators
    if "highly" in t and "oscill" in t:
        return 2
    if "oscillating sp" in t:
        return 2
    if "oscill" in t:
        return 2
    
    # Maybe oscillatory
    if "intermitt" in t:
        return 1
    if "small oscill" in t:
        return 1
    if "some ripple" in t or "ripples" in t:
        return 1
    if "variance" in t:
        return 1
    
    # Not oscillation
    if "no" == t or "no oscillation" in t:
        return 0
    if "noisy" in t:
        return 0
    
    return 0  # default


def map_noise(text):
    t = clean_text(text)
    if t == "":
        return 0
    if "yes" == t:
        return 2
    if "could" in t or "may" in t:
        return 1
    if "no" in t:
        return 0
    return 0


def map_disturbance(text):
    t = clean_text(text)
    if t == "":
        return 0
    if "yes" in t or "disturb" in t:
        return 2
    if "may" in t:
        return 1
    if "no" in t:
        return 0
    return 0


def map_stiction(text):
    t = clean_text(text)
    if t == "":
        return 0.3
    if "doesn't seem" in t or "no stiction" in t:
        return 0.1
    if "possible" in t or "maybe" in t or "friction" in t:
        return 0.6
    if "stiction" in t or "sticky" in t:
        return 0.9
    return 0.3


def extract_all_labels(df):
    df["osc_label"] = df["Oscillatory?"].apply(map_oscillation)
    df["noise_label"] = df["Noisy"].apply(map_noise)
    df["dist_label"] = df["Disturbance"].apply(map_disturbance)
    df["stiction_prob"] = df["Sticky Valve probability "].apply(map_stiction)
    return df


In [11]:

labels_df = extract_all_labels(loop_desc)

labels_df.head()


KeyError: 'Stiction'

In [ ]:
def infer_fs(time_series):
    """Infer sampling frequency from timestamp column"""
    try:
        ts = pd.to_datetime(time_series)
        diffs = ts.diff().dropna().dt.total_seconds().values
        if len(diffs) == 0:
            return 1.0
        dt = np.median(diffs)
        return 1.0 / dt if dt > 0 else 1.0
    except:
        return 1.0


def autocorr_peak_strength(x, maxlag=300):
    """Compute maximum non-zero autocorrelation peak."""
    x = x - np.mean(x)
    r = np.correlate(x, x, mode='full')
    mid = len(r) // 2
    ac = r[mid+1: mid+1+maxlag] / (r[mid] if r[mid] != 0 else 1.0)
    return np.max(ac[2:]) if len(ac) > 5 else 0.0


def sliding_slopes(op, pv, window):
    """Compute slopes of OP→PV for stiction detection."""
    slopes = []
    n = len(op)
    for i in range(0, n-window+1, window):
        X = op[i:i+window].reshape(-1, 1)
        y = pv[i:i+window]
        if np.std(X) < 1e-8:
            slopes.append(0.0)
            continue
        reg = LinearRegression().fit(X, y)
        slopes.append(reg.coef_[0])
    return np.array(slopes)
